In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.metrics import brier_score


In [2]:
#=== Load the data ===#

train_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/train.csv")
train_df = train_df.drop("event_id", axis=1)
test_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/test.csv")

meta_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/metaData.csv")

submission = pd.read_csv("WiDSWorldWide_GlobalDathon26/sample_submission.csv")

In [3]:
#=== Prepare the data ===#

# Drop target columns
X = train_df.drop(["event", "time_to_hit_hours"], axis=1)

y_event = train_df["event"].astype(bool)
y_time = train_df["time_to_hit_hours"]

# Structured survival array (required format)
y = np.array(list(zip(y_event, y_time)),
             dtype=[("event", bool), ("time", float)])


X_test = test_df.drop(["event_id"], axis=1)

In [4]:
#=== 5-fold cross-validation setup===#
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store test predictions from each fold
test_preds_folds = []

In [19]:
#=== Cross-validation training ===#
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

    print(f"\n===== Fold {fold+1} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Scale (fit ONLY on training fold)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # ---- STEP 1: Fit regularization path ---- #
    cox_path = CoxnetSurvivalAnalysis(
        l1_ratio=1e-5,
        alpha_min_ratio=0.01,
        n_alphas=50
    )

    cox_path.fit(X_train_scaled, y_train)

    # ---- STEP 2: Select best alpha via validation C-index ---- #
    best_alpha = None
    best_score = -1

    for alpha in cox_path.alphas_:
        risk_scores = cox_path.predict(X_val_scaled, alpha=alpha)

        c_index = concordance_index_censored(
            y_val["event"],
            y_val["time"],
            risk_scores
        )[0]

        if c_index > best_score:
            best_score = c_index
            best_alpha = alpha

    print("Best alpha:", best_alpha)
    print("Validation C-index:", best_score)

    # ---- STEP 3: Refit with chosen alpha + baseline ---- #
    cox_best = CoxnetSurvivalAnalysis(
        l1_ratio=1e-5,
        alphas=[best_alpha],
        fit_baseline_model=True
    )

    cox_best.fit(X_train_scaled, y_train)

    # ---- STEP 4: Predict survival for test set ---- #
    surv_funcs_test = cox_best.predict_survival_function(X_test_scaled)

    times = np.array([12.0, 24.0, 48.0, 72.0])
    fold_probs = []

    for fn in surv_funcs_test:
        surv_at_times = np.interp(times, fn.x, fn.y)
        probs = 1 - surv_at_times  # convert to hit probability
        probs = np.clip(probs, 0.01, 0.99)  # mild calibration stabilizer
        fold_probs.append(probs)

    test_preds_folds.append(np.array(fold_probs))


===== Fold 1 =====
Best alpha: 19040.950176142476
Validation C-index: 0.9203020134228188

===== Fold 2 =====
Best alpha: 22926.590391835703
Validation C-index: 0.9386666666666666

===== Fold 3 =====
Best alpha: 21277.4739635413
Validation C-index: 0.7788990825688074

===== Fold 4 =====
Best alpha: 24419.749123093196
Validation C-index: 0.9056399132321041

===== Fold 5 =====
Best alpha: 22806.45719262243
Validation C-index: 0.9340796019900498


In [20]:
# ---- FINAL TEST PREDICTION (average folds) ---- #
final_test_preds = np.mean(test_preds_folds, axis=0)

print("\nFinished CV training.")


Finished CV training.


In [21]:
# Average predictions across folds


test_preds = np.mean(test_preds_folds, axis=0)

probs_12h = test_preds[:, 0]
probs_24h = test_preds[:, 1]
probs_48h = test_preds[:, 2]
probs_72h = test_preds[:, 3]

In [22]:
#=== c-score ===#

# risk scores (higher = more risk)
risk_scores = cox_best.predict(X_val_scaled)

c_index = concordance_index_censored(
    y_val["event"],
    y_val["time"],
    risk_scores
)[0]

print("C-index:", c_index)

C-index: 0.9340796019900498


In [23]:
#=== Brier score ===#

max_time = y_val["time"].max()

times_eval = np.array([24.0, 48.0, 72.0])
times_eval = times_eval[times_eval < max_time]

print("Using horizons:", times_eval)


# Predict survival curves
surv_funcs_val = cox_best.predict_survival_function(X_val_scaled)

# Convert to survival probability matrix
surv_matrix = []

for fn in surv_funcs_val:
    surv_at_times = np.interp(times_eval, fn.x, fn.y)
    surv_matrix.append(surv_at_times)

surv_matrix = np.array(surv_matrix)

_, brier_scores = brier_score(
    y_train,     # training data for censor distribution
    y_val,
    surv_matrix,
    times_eval
)

for t, score in zip(times_eval, brier_scores):
    print(f"Brier @{t}h:", score)


weights = {24.0: 0.3, 48.0: 0.4, 72.0: 0.3}

weighted_brier = float(np.sum([
    weights[t] * score
    for t, score in zip(times_eval, brier_scores)
]))

print("Weighted Brier:", weighted_brier)

Using horizons: [24. 48.]
Brier @24.0h: 0.19657888349540667
Brier @48.0h: 0.19164189576853266
Weighted Brier: 0.13563042335603506


In [24]:
hybrid_score = 0.3 * c_index + 0.7 * (1 - weighted_brier)

print("Hybrid Score:", hybrid_score)

Hybrid Score: 0.8852825842477903


In [13]:
print("C-index:", c_index)
print("Weighted Brier:", weighted_brier)
print("Hybrid Score:", hybrid_score)

C-index: 0.9340796019900498
Weighted Brier: 0.13563042335603506
Hybrid Score: 0.8852825842477903


In [ ]:
sub = pd.DataFrame({
    "event_id": test_df["event_id"],
    "prob_12h": probs_12h,
    "prob_24h": probs_24h,
    "prob_48h": probs_48h,
    "prob_72h": probs_72h,
})

sub.to_csv("submission.csv", index=False)
print("Submission saved.")